# Credit card default prediction (Taiwan, 2005)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binomtest, chi2_contingency

In [ ]:
DATA_PATH = "../UCI_Credit_Card.csv"
FEATURES_PATH = "credit_features.csv"
TARGET = "default"
PAY_STATUS_COLUMNS = [f"PAY_{month}" for month in range(1, 7)]
BILL_COLUMNS = [f"BILL_AMT{month}" for month in range(1, 7)]
PAYMENT_COLUMNS = [f"PAY_AMT{month}" for month in range(1, 7)]
STATUS_COLUMNS = [f"STATUS_{month}" for month in range(1, 7)]
DELAY_COLUMNS = [f"DELAY_{month}" for month in range(1, 7)]
UTILIZATION_CAP = 2
AGE_OF_LOWEST_RISK = 33

## 1. Introduction

Credit card clients in Taiwan, with data from April to September 2005. The goal is to predict whether a client defaults the following month (`default`, 1 = default).

This notebook covers loading, cleaning, exploratory analysis and feature engineering, and answers the first guiding question: **how does default vary across demographic variables?** Modelling is in `02_Modeling_CC` and evaluation in `03_Evaluation_CC`.

## 2. Data loading

We rename `PAY_0` to `PAY_1` so that the indices of the three monthly series match (1 = September, 6 = April), and the target to `default` to remove the dots from its name. The original DataFrame `credit_raw` is not modified.

In [ ]:
credit_raw = pd.read_csv(DATA_PATH)
credit = credit_raw.rename(columns={"default.payment.next.month": TARGET, "PAY_0": "PAY_1"})
credit.head()

## 3. Cleaning

We apply the author's corrected coding on a copy, `credit_clean`:

| Variable | Value | Label |
|---|---|---|
| `EDUCATION` | 1 / 2 / 3 | postgraduate / university / high school |
| | 0, 4, 5, 6 | other |
| `MARRIAGE` | 1 / 2 / 3 / 0 | married / single / divorced / other |
| `PAY_*` | -2 / -1 / 0 | unused / paid in full / revolving |
| | 1 … 9 | months of delay |

All rows are kept. Negative and extreme amounts are handled in feature engineering.

In [ ]:
sex_labels = {1: "man", 2: "woman"}
education_labels = {
    1: "postgraduate", 2: "university", 3: "high_school",
    0: "other", 4: "other", 5: "other", 6: "other",
}
marriage_labels = {1: "married", 2: "single", 3: "divorced", 0: "other"}

credit_clean = credit.copy()
credit_clean["SEX"] = credit["SEX"].map(sex_labels)
credit_clean["EDUCATION"] = credit["EDUCATION"].map(education_labels)
credit_clean["MARRIAGE"] = credit["MARRIAGE"].map(marriage_labels)

Each `PAY_*` column mixes the type of credit usage (-2, -1, 0) and the months of delay (1 to 9). We split it into two variables that are simpler than an 11-level categorical:

- `STATUS_*`: type of usage (unused, paid in full, revolving or past due).
- `DELAY_*`: months of delay, 0 if the code is ≤ 0.

In [ ]:
status_labels = {-2: "unused", -1: "paid_full", 0: "revolving", 1: "past_due"}

for month in range(1, 7):
    pay_column = f"PAY_{month}"
    credit_clean[f"STATUS_{month}"] = credit[pay_column].clip(upper=1).map(status_labels)
    credit_clean[f"DELAY_{month}"] = credit[pay_column].clip(lower=0)

credit_clean = credit_clean.drop(columns=PAY_STATUS_COLUMNS)

## 4. Exploratory analysis

### 4.1 Target variable

In [ ]:
credit_clean[TARGET].value_counts(normalize=True).round(3)

22.1% of clients default. A model that always predicted "no default" would reach an accuracy of 77.9% without detecting a single default. That is why the main metrics will be PR-AUC, ROC-AUC and the recall and precision of class 1, and all splits will be stratified.

### 4.2 Default by group

We answer the first guiding question: how does default vary across demographic variables? For each group we compute the default rate with a 95% Wilson confidence interval. Age and credit limit are grouped into bands only for this analysis, in a separate copy (`credit_eda`).

In [ ]:
def default_rate_with_ci(data, column):
    rows = []
    for group, group_target in data.groupby(column, observed=True)[TARGET]:
        test_result = binomtest(int(group_target.sum()), len(group_target))
        interval = test_result.proportion_ci(method="wilson")
        rows.append({
            column: group,
            "clients": len(group_target),
            "default_rate": group_target.mean(),
            "ci_lower": interval.low,
            "ci_upper": interval.high,
        })
    return pd.DataFrame(rows).set_index(column).round(3)

In [ ]:
overall_default_rate = credit_clean[TARGET].mean()


def plot_default_rate(rate_table, title, xlabel):
    lower_error = rate_table["default_rate"] - rate_table["ci_lower"]
    upper_error = rate_table["ci_upper"] - rate_table["default_rate"]
    ax = rate_table["default_rate"].plot(
        kind="bar",
        yerr=np.array([lower_error, upper_error]),
        capsize=4,
        rot=0,
        label="Group rate (95% CI)",
    )
    ax.axhline(overall_default_rate, color="grey", linestyle="--", label="Overall rate")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Default rate")
    ax.legend()
    plt.show()

In [ ]:
age_bins = [20, 25, 30, 35, 40, 45, 50, 60, 80]
limit_bins = [0, 50_000, 100_000, 200_000, 300_000, 500_000, 1_000_000]
limit_labels = ["≤50k", "50k–100k", "100k–200k", "200k–300k", "300k–500k", ">500k"]

credit_eda = credit_clean.assign(
    AGE_BAND=pd.cut(credit_clean["AGE"], bins=age_bins),
    LIMIT_BAND=pd.cut(credit_clean["LIMIT_BAL"], bins=limit_bins, labels=limit_labels),
)

### 4.3 Education

In [ ]:
education_order = ["postgraduate", "university", "high_school", "other"]
education_default = default_rate_with_ci(credit_eda, "EDUCATION").loc[education_order]
plot_default_rate(education_default, "Default rate by education level", "Education level")

The lower the education level, the higher the default rate: 19.2% for postgraduate, 23.7% for university and 25.2% for high school. The "other" group (7.1%) should not be read as an education level: it pools several residual codes, and its unusual rate probably reflects how those clients were recorded.

### 4.4 Age

In [ ]:
age_default = default_rate_with_ci(credit_eda, "AGE_BAND")
plot_default_rate(age_default, "Default rate by age band", "Age (years)")

The relationship is U-shaped. Default peaks among the youngest clients (26.7% between 21 and 25), is lowest between 26 and 35 (19–20%) and rises again from 45 (25%). A logistic regression using age as a linear variable will not capture this well, so we will need bands or a quadratic term.

### 4.5 Credit limit

It is not a demographic variable, but the bank sets the limit based on its own assessment of the client, and it may explain part of the previous differences.

In [ ]:
limit_default = default_rate_with_ci(credit_eda, "LIMIT_BAND")
plot_default_rate(
    limit_default, "Default rate by credit limit band", "Credit limit (NT$)"
)

This is the clearest gradient: default falls steadily from 31.8% for limits up to NT$50,000 to 11.2% above NT$500,000. It should be read as a correlation, not a cause. The bank gives high limits to clients it already considers creditworthy, so the limit carries its prior risk assessment; raising a client's limit would not reduce their risk.

### 4.6 Sex and marital status

These are the two weakest associations, so we only show their rates with confidence intervals.

In [ ]:
pd.concat({
    "SEX": default_rate_with_ci(credit_eda, "SEX"),
    "MARRIAGE": default_rate_with_ci(credit_eda, "MARRIAGE"),
})

Men default 3.4 points more than women (24.2% vs 20.8%), with non-overlapping intervals. Married clients default more than single ones (23.5% vs 20.9%). Divorced clients have the highest rate (26.0%), but with a wide interval (21.5–31.1%), and "other" (54 clients) is not reliable.

### 4.7 Independence test

With 30,000 clients, even small differences are significant in the chi-square test, so Cramér's V (from 0 to 1) is what measures whether the association matters.

In [ ]:
test_rows = []
for column in ["SEX", "EDUCATION", "MARRIAGE", "AGE_BAND", "LIMIT_BAND"]:
    contingency = pd.crosstab(credit_eda[column], credit_eda[TARGET])
    chi2, p_value, degrees_of_freedom, _ = chi2_contingency(contingency)
    cramers_v = np.sqrt(chi2 / contingency.to_numpy().sum())
    test_rows.append({
        "variable": column,
        "chi2": chi2,
        "df": degrees_of_freedom,
        "p_value": p_value,
        "cramers_v": cramers_v,
    })

test_results = pd.DataFrame(test_rows).set_index("variable")
test_results.sort_values("cramers_v", ascending=False).round({"chi2": 1, "cramers_v": 3})

All differences are significant (p < 0.001), but their size is modest. Credit limit shows the strongest association (V = 0.16), far ahead of education (0.07) and age (0.06). Sex (0.04) and marital status (0.03) are barely associated with default.

### 4.8 Conclusions on the first guiding question

- Default varies significantly across all groups, but **demographic differences are small** (Cramér's V from 0.03 to 0.07): high school vs postgraduate (25.2% vs 19.2%), under 25 and over 60 vs 31–35 (≈ 27% vs 19.4%), and men vs women (24.2% vs 20.8%).
- **Credit limit shows the largest gap** (31.8% vs 11.2%; V = 0.16), but the bank sets it based on its own risk assessment, so it is a correlation, not a cause.
- `SEX`, `AGE` and `MARRIAGE` are sensitive attributes that many jurisdictions forbid in credit decisions.

## 5. Feature engineering

We summarise each client's six-month history in variables with a business meaning. They are computed row by row with data from April to September, the information available when predicting October's default, so there is no information leakage. We work on a new copy, `credit_features`.

- **Utilisation** (`UTILIZATION_*`): bill / limit in September, mean and maximum, capped at 2 (`UTILIZATION_CAP`). Credit balances (negative bills) are kept.
- **Payment ratio** (`PAYMENT_RATIO_*`): payment in a month / bill of the previous month, capped at 1, and 1 when there was nothing to pay.
- **Delays and type of usage**: months with delay, worst delay and months in each type of usage.
- **Age** (`AGE_SQUARED_DEVIATION`): squared distance to age 33, so the logistic regression can capture the U shape.
- **Debt, amounts and activity**: debt trend (September − April, over the limit), logarithms of the limit and of payments, and `INACTIVE` (no bills or payments in six months).

In [ ]:
credit_features = credit_clean.copy()
utilization = credit_clean[BILL_COLUMNS].div(credit_clean["LIMIT_BAL"], axis=0)
utilization = utilization.clip(upper=UTILIZATION_CAP)

credit_features["UTILIZATION_LAST"] = utilization["BILL_AMT1"]
credit_features["UTILIZATION_MEAN"] = utilization.mean(axis=1)
credit_features["UTILIZATION_MAX"] = utilization.max(axis=1)

payment_ratios = pd.DataFrame(index=credit_clean.index)
for month in range(1, 6):
    payment = credit_clean[f"PAY_AMT{month}"]
    previous_bill = credit_clean[f"BILL_AMT{month + 1}"]
    ratio = (payment / previous_bill).where(previous_bill > 0, 1)
    payment_ratios[month] = ratio.clip(upper=1)

credit_features["PAYMENT_RATIO_LAST"] = payment_ratios[1]
credit_features["PAYMENT_RATIO_MEAN"] = payment_ratios.mean(axis=1)

In [ ]:
credit_features["MONTHS_DELAYED"] = (credit_clean[DELAY_COLUMNS] > 0).sum(axis=1)
credit_features["WORST_DELAY"] = credit_clean[DELAY_COLUMNS].max(axis=1)

status_feature_names = {
    "revolving": "MONTHS_REVOLVING",
    "paid_full": "MONTHS_PAID_FULL",
    "unused": "MONTHS_NO_USE",
}
for status, feature_name in status_feature_names.items():
    credit_features[feature_name] = (credit_clean[STATUS_COLUMNS] == status).sum(axis=1)

age_deviation = credit_clean["AGE"] - AGE_OF_LOWEST_RISK
credit_features["AGE_SQUARED_DEVIATION"] = age_deviation ** 2

In [ ]:
bill_change = credit_clean["BILL_AMT1"] - credit_clean["BILL_AMT6"]
debt_trend = bill_change / credit_clean["LIMIT_BAL"]
credit_features["DEBT_TREND"] = debt_trend.clip(-UTILIZATION_CAP, UTILIZATION_CAP)

credit_features["LOG_LIMIT"] = np.log(credit_clean["LIMIT_BAL"])
credit_features["LOG_PAYMENT_LAST"] = np.log1p(credit_clean["PAY_AMT1"])
credit_features["LOG_PAYMENT_MEAN"] = np.log1p(credit_clean[PAYMENT_COLUMNS].mean(axis=1))

no_activity = (credit_clean[BILL_COLUMNS + PAYMENT_COLUMNS] == 0).all(axis=1)
credit_features["INACTIVE"] = no_activity.astype(int)

### 5.1 Redundancy among the new variables

If two variables say almost the same thing, keeping both makes the logistic regression coefficients unstable. We look for pairs with an absolute correlation above 0.85.

In [ ]:
candidate_numeric_features = [
    "AGE", "AGE_SQUARED_DEVIATION", "LOG_LIMIT",
    "DELAY_1", "MONTHS_DELAYED", "WORST_DELAY", "MONTHS_PAID_FULL", "MONTHS_NO_USE",
    "UTILIZATION_LAST", "UTILIZATION_MEAN", "UTILIZATION_MAX",
    "PAYMENT_RATIO_LAST", "PAYMENT_RATIO_MEAN", "DEBT_TREND",
    "LOG_PAYMENT_LAST", "LOG_PAYMENT_MEAN", "INACTIVE",
]
correlations = credit_features[candidate_numeric_features].corr().abs()
upper_triangle = np.triu(np.ones(correlations.shape, dtype=bool), k=1)
correlated_pairs = correlations.where(upper_triangle).stack()
correlated_pairs[correlated_pairs > 0.85].sort_values(ascending=False).round(2)

There are two redundant families: the three utilisation variants (correlations from 0.91 to 0.94) and the two payment ratio variants (0.88). From each family we keep the mean, which summarises the six months and is less sensitive to an unusual month.

Pairwise correlation does not detect a variable that is explained by a combination of several others. For that we compute the variance inflation factor (VIF) of the remaining variables. A VIF above 5 indicates redundancy.

In [ ]:
correlated_features = ["UTILIZATION_LAST", "UTILIZATION_MAX", "PAYMENT_RATIO_LAST"]
remaining_features = [
    feature for feature in candidate_numeric_features if feature not in correlated_features
]
remaining_correlations = credit_features[remaining_features].corr().to_numpy()
variance_inflation = pd.Series(
    np.diag(np.linalg.inv(remaining_correlations)), index=remaining_features, name="VIF"
)
variance_inflation.sort_values(ascending=False).head(5).round(2)

`PAYMENT_RATIO_MEAN` (15.7), `MONTHS_PAID_FULL` (8.6) and `MONTHS_NO_USE` (8.1) exceed the threshold: clients who pay in full or do not use the card have, by construction, a payment ratio of 1. We drop `MONTHS_NO_USE` and all VIFs fall below 3.7. This way we keep the payment ratio, which has a direct business reading.

### 5.2 Selection and export

The summary variables replace the 18 raw monthly columns and `STATUS_2`–`STATUS_6`. We also drop `MONTHS_REVOLVING`, which is redundant (together with the other three usage counts it always adds up to 6), and `MONTHS_NO_USE`, because of its VIF. The model starts from 17 variables:

| Family | Variables |
|---|---|
| Demographic | `SEX`, `EDUCATION`, `MARRIAGE`, `AGE`, `AGE_SQUARED_DEVIATION` |
| Credit | `LOG_LIMIT` |
| Current status | `STATUS_1`, `DELAY_1` |
| Delay history | `MONTHS_DELAYED`, `WORST_DELAY` |
| Type of usage | `MONTHS_PAID_FULL` |
| Usage and payment | `UTILIZATION_MEAN`, `PAYMENT_RATIO_MEAN`, `DEBT_TREND`, `LOG_PAYMENT_LAST`, `LOG_PAYMENT_MEAN` |
| Activity | `INACTIVE` |

We save them, together with the target, to `credit_features.csv`, which `02_Modeling_CC` reads.

In [ ]:
numeric_features = [
    feature for feature in remaining_features if feature != "MONTHS_NO_USE"
]
categorical_features = ["SEX", "EDUCATION", "MARRIAGE", "STATUS_1"]
model_features = categorical_features + numeric_features

pd.Series({"categorical": len(categorical_features), "numeric": len(numeric_features)})

In [ ]:
credit_features[model_features + [TARGET]].to_csv(FEATURES_PATH, index=False)
credit_features[model_features + [TARGET]].shape